# Accuracy vs Cost Tradeoff — xRouteBench

Two views of the same 18-model candidate pool from `llmrouter_generic`
(train + test):

- **Chart 1 — Model vs Accuracy:** average task score per candidate.
- **Chart 2 — Model vs Cost:** average per-query dollar cost per candidate
  (from `llm_candidates` pricing × token usage).
- **Chart 3 — Cost vs Accuracy scatter (business case):** shows where the
  "cheap and good" models sit — the Pareto frontier a router exploits.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
RAW = root / "data/raw"

train = pd.read_parquet(RAW / "llmrouter_generic/train-000.parquet")
test = pd.read_parquet(RAW / "llmrouter_generic/test-000.parquet")
routing = pd.concat([train, test], ignore_index=True)
cand = pd.read_parquet(RAW / "llm_candidates/train-000.parquet")

price = cand.set_index("model_name")[["input_price_per_1m", "output_price_per_1m"]]
df = routing.merge(price, left_on="model_name", right_index=True, how="left")
df["cost_usd"] = (
    df["input_tokens"] * df["input_price_per_1m"]
    + df["output_tokens"] * df["output_price_per_1m"]
) / 1e6

agg = (
    df.groupby("model_name")
    .agg(accuracy=("performance", "mean"), mean_cost=("cost_usd", "mean"), rows=("performance", "size"))
    .reset_index()
)
print(f"rows={len(df)} | candidates={len(agg)}")
print(agg.sort_values("accuracy", ascending=False).to_string(index=False))

## Chart 1 — Model vs Accuracy

In [ ]:
plt.figure(figsize=(12, 5))
by_acc = agg.sort_values("accuracy").reset_index(drop=True)
plt.barh(by_acc["model_name"], by_acc["accuracy"], color="#3b82f6")
plt.xlabel("Mean accuracy (task score)")
plt.title("Chart 1 — Candidate model vs accuracy")
plt.tight_layout()
plt.show()

## Chart 2 — Model vs Cost

In [ ]:
plt.figure(figsize=(12, 5))
by_cost = agg.sort_values("mean_cost").reset_index(drop=True)
plt.barh(by_cost["model_name"], by_cost["mean_cost"], color="#10b981")
plt.xlabel("Mean per-query cost (USD)")
plt.title("Chart 2 — Candidate model vs per-query cost")
plt.tight_layout()
plt.show()

## Chart 3 — Business case: cost vs accuracy tradeoff

Quadrants:
- **top-left = ideal** (high accuracy, low cost) → the models a router should use.
- bottom-right = expensive but not better (poor value).
The router wants to *pick on the Pareto frontier*: accuracy you need at the
lowest cost.

In [ ]:
plt.figure(figsize=(11, 7))
plt.scatter(agg["mean_cost"], agg["accuracy"], s=70, color="#ef4444")
for _, r in agg.iterrows():
    plt.annotate(r["model_name"], (r["mean_cost"], r["accuracy"]),
                 textcoords="offset points", xytext=(6, 6), fontsize=8)
plt.axvline(agg["mean_cost"].median(), color="gray", ls=":", lw=1)
plt.axhline(agg["accuracy"].median(), color="gray", ls=":", lw=1)
plt.xlabel("Mean per-query cost (USD)")
plt.ylabel("Mean accuracy")
plt.title("Chart 3 — Cost vs accuracy: the routing business case")
plt.tight_layout()
plt.show()